# Fine-tuned Apertus 8B

## Generate text using the baseline model

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
import torch
from transformers import pipeline

/storage/homefs/as23z124/.conda/envs/CAS_NLP_M4/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
prompt = "Once upon a time there was a fairy"

In [20]:
pipe_base = pipeline(
    "text-generation",
    model="swiss-ai/Apertus-8B-2509",
    device="cuda",
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.33s/it]
Device set to use cuda


In [ ]:

baseline = pipe_base(
    prompt,
    max_new_tokens=200,       # maximum length of generated continuation
    min_new_tokens=100,       # enforce a minimum generation length
    do_sample=True,           # enable stochastic sampling instead of greedy decoding
    temperature=0.9,          # controls randomness (lower = more deterministic, higher = more creative) 1.0 is standard. 0.9 = slightly creative but still coherent
    top_p=0.95,               # nucleus sampling: restricts sampling to the most likely tokens (95% mass)
    #repetition_penalty=1.1,  # (optional) discourages repeating previously generated tokens
    #no_repeat_ngram_size=3   # (optional) prevents exact repetition of 3-token sequences
)

baseline = baseline[0]["generated_text"]
print(baseline)

CUDA-fused xIELU not available (No module named 'xielu') – falling back to a Python version.
For CUDA xIELU (experimental), `pip install git+https://github.com/nickjbrowning/XIELU`
Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.42s/it]
Device set to use cuda


Once upon a time there was a fairy who lived in a
cloud. She was so light that she could fly with the
wildest wind; and the wind made her clothes of
cloud so thin that a breath would blow them away.
But she was a powerful fairy, who could make the
wind blow a hundred times stronger, if she chose.
She could make rain and hail, or snow and sleet;
and she could send the lightning and thunder too.
Besides all this, she could change herself into
whatever form she liked. For instance, she could
change herself into a beautiful girl or a very
ugly one. You would never know which it was,
if she did not choose to tell you.
Sometimes she would make herself very tall
and very slender, with long hair and eyes like
fire; but at other times she would look like an old
woman with gray hair and wrinkles, and eyes
which were dull and weak like the eyes of an owl.
She was very


---

# Import the checkpoints

In [3]:
from pathlib import Path

# Convert to list so it can be reused
checkpoint_paths = sorted(Path("./models/apertus_fairytales").glob("checkpoint-epoch*"))

for checkpoint in checkpoint_paths:
    print(checkpoint)

models/apertus_fairytales/checkpoint-epoch1
models/apertus_fairytales/checkpoint-epoch2
models/apertus_fairytales/checkpoint-epoch3
models/apertus_fairytales/checkpoint-epoch4
models/apertus_fairytales/checkpoint-epoch5


In [7]:
import json
from datetime import datetime
from pathlib import Path
import gc

# Create directory for text collections if it doesn't exist
collection_dir = Path("text_collections_apertus_fairytales")
collection_dir.mkdir(exist_ok=True)

# Generate text from each checkpoint
generated_texts = []
for checkpoint_path in checkpoint_paths:
    # Extract checkpoint epoch number
    checkpoint_number = checkpoint_path.name.split('-')[1]
    
    print(f"Loading checkpoint-{checkpoint_number}...")
    
    # Load tokenizer and model for this checkpoint WITH MEMORY-EFFICIENT SETTINGS
    tokenizer_checkpoint = AutoTokenizer.from_pretrained(checkpoint_path)
    model_checkpoint = AutoModelForCausalLM.from_pretrained(
        checkpoint_path,
        torch_dtype=torch.bfloat16,  # Use bfloat16 to save memory (same as training)
        device_map="auto"             # Automatic device placement
    )
    
    # Create pipeline (don't specify device when using device_map="auto")
    pipe_checkpoint = pipeline(
        "text-generation",
        model=model_checkpoint,
        tokenizer=tokenizer_checkpoint,
    )
    
    # Generate text
    ft_output = pipe_checkpoint(
        prompt,
        max_new_tokens=200,
        min_new_tokens=100,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        #repetition_penalty=1.1,
        #no_repeat_ngram_size=3
    )
    
    # Extract generated text and store with variable name pattern
    finetuned_text = ft_output[0]["generated_text"]
    
    # Store in a variable named like finetuned_16655, finetuned_33310, etc.
    globals()[f'finetuned_{checkpoint_number}'] = finetuned_text
    
    print(f"Checkpoint: checkpoint-{checkpoint_number}")
    print(finetuned_text)
    print("\n" + "="*80 + "\n")
    
    generated_texts.append({
        "checkpoint": f"checkpoint-{checkpoint_number}",
        "checkpoint_number": checkpoint_number,
        "checkpoint_path": str(checkpoint_path),
        "text": finetuned_text,
        "prompt": prompt,
        "timestamp": datetime.now().isoformat()
    })
    
    # Clean up to free GPU memory before loading next checkpoint
    del model_checkpoint
    del pipe_checkpoint
    del tokenizer_checkpoint
    torch.cuda.empty_cache()
    gc.collect()
    print(f"✓ Cleaned up memory for checkpoint-{checkpoint_number}\n")

# Save to JSON with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = collection_dir / f"generated_stories_{timestamp}.json"

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(generated_texts, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved {len(generated_texts)} generated texts to: {output_file}")
print(f"✓ Created variables: " + ", ".join([f"finetuned_{item['checkpoint_number']}" for item in generated_texts]))

Loading checkpoint-epoch1...


Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.81s/it]
Device set to use cuda:0


Checkpoint: checkpoint-epoch1
Once upon a time there was a fairy named Paribanou, who was the wife of the Sultan, and mother of the Princess Badoura. As the Sultan was one day playing a game of chess in the palace courtyard with a stranger, who was a good chess-player, the stranger saw the Princess Badoura, who was very lovely, and was struck with her beauty, and he took up his chessmen, and turned away. The Sultan’s wife saw this, and said to her husband: “My Lord, what is the reason that the stranger, who was playing with you, went away without bidding me farewell, as was the custom of the country?” “He went away,” said the Sultan, “because he was afraid of losing the game.” “It was not that,” said the stranger, “for I had not played ten moves with him before I saw in the next room a Princess so lovely, that I should be very unhappy if I had not the good fortune to see her face.” The Sultan, who was a


✓ Cleaned up memory for checkpoint-epoch1

Loading checkpoint-epoch2...


Loading checkpoint shards: 100%|██████████| 4/4 [00:22<00:00,  5.66s/it]
Device set to use cuda:0


Checkpoint: checkpoint-epoch2
Once upon a time there was a fairy who was the guardian of a young man called Prince Darling. He was not really the son of a king or a knight, but the child of a poor woman who was so beautiful and good that she was known as the Queen of the Butterflies. This Fairy used to take him into a room filled with flowers where she gave him lessons in everything that was kind and beautiful. But one day there came an earthquake which threw down a wall of the palace, and Prince Darling and the Queen of the Butterflies were buried under the ruins and killed, and the Fairy herself was carried away in a flood. The Prince was the only child of his mother, and the Fairy was his only friend, so you can imagine how sorry he was when they were gone. He made up his mind that he would go out into the world to seek his fortune, but he was afraid lest some one should rob him, and he thought if he could find some one to whom he could give his friendship, he might help


✓ Cleaned

...

Checkpoint: checkpoint-epoch3
Once upon a time there was a fairy called Puss-in-Boots, who was very handsome and smart, and who had two children, a son and a daughter, exactly like himself. Unfortunately the little son died when he was only seven years old; and this made both the Fairy and his sister very sad. For the son had been very good and lovely, and his loss was a great affliction to them. But the Fairy, instead of giving way to her grief, set herself to work to make her daughter a princess, and to try and make up for the loss of her son. She sent the little girl to the best schools, and taught her how to be graceful and to play all sorts of pretty games. Besides this, she made up her mind not to leave her daughter entirely without a brother, and by her magic arts she succeeded in transforming a fine large grey cat into a little boy. This boy was as handsome and good as he could be, but he never grew at all, and when he was ten


✓ Cleaned up memory for checkpoint-epoch3

Loadin

Loading checkpoint shards: 100%|██████████| 4/4 [00:23<00:00,  5.90s/it]
Device set to use cuda:0


Checkpoint: checkpoint-epoch4
Once upon a time there was a fairy who loved very much to try her powers upon poor, human mortals, and she was always thinking of new and strange ways to trouble them. But at last a new idea came into her head, which beat all the others, and so she determined to make a man of wax, who, if he was only properly treated, should be almost as good as the real thing. So, when she went into the garden one day to gather the flowers she needed for this great work, she looked about to see which of her many flowers would be the most useful to her. The violet, she thought, was too humble; the rose too haughty; the tulip too conceited; the lily too proud, and so on till she had tried them all. But as soon as she came to the buttercup, it did not need her to try it, for it jumped right up and asked her if she wanted it. “You certainly do,” said the fairy. “You


✓ Cleaned up memory for checkpoint-epoch4

Loading checkpoint-epoch5...


Loading checkpoint shards: 100%|██████████| 4/4 [00:23<00:00,  5.89s/it]
Device set to use cuda:0


Checkpoint: checkpoint-epoch5
Once upon a time there was a fairy who loved to sport in the air among the clouds and over the tops of the lofty mountains. She loved to wander about the earth and to mix with mortals, listening to their troubles and their happiness, and to their talk and laughter. She could change herself into a beautiful woman or a beautiful bird, at will. She had been a beautiful maiden once, but the enchantment of the elves had changed her into a bird. But she would change herself back into a beautiful woman by and by, when she had found a worthy lover. In the spring she loved to choose out a pretty spot, on the top of a high mountain, or a hill-side, where the sun shone brightly, and in June she would lay her golden eggs and set them to be hatched. When the little birds were grown up they would fly away, each one to some mountain or hill-side, to lay her own eggs, and then they would return to the same spot, or to some neighboring spot,


✓ Cleaned up memory for check

In [8]:
generated_texts

[{'checkpoint': 'checkpoint-epoch1',
  'checkpoint_number': 'epoch1',
  'checkpoint_path': 'models/apertus_fairytales/checkpoint-epoch1',
  'text': 'Once upon a time there was a fairy named Paribanou, who was the wife of the Sultan, and mother of the Princess Badoura. As the Sultan was one day playing a game of chess in the palace courtyard with a stranger, who was a good chess-player, the stranger saw the Princess Badoura, who was very lovely, and was struck with her beauty, and he took up his chessmen, and turned away. The Sultan’s wife saw this, and said to her husband: “My Lord, what is the reason that the stranger, who was playing with you, went away without bidding me farewell, as was the custom of the country?” “He went away,” said the Sultan, “because he was afraid of losing the game.” “It was not that,” said the stranger, “for I had not played ten moves with him before I saw in the next room a Princess so lovely, that I should be very unhappy if I had not the good fortune to s

## Self-BLEU

Self-BLEU measures how similar generated samples are to each other. Lower Self-BLEU is better for creativity/diversity 

In [9]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [10]:
def self_bleu(texts, n_gram=2, return_std=True):
    """
    Compute Self-BLEU to measure similarity among generated texts.
    Optionally returns standard deviation across texts.

    Args:
        texts (list[str]): List of generated texts from the same model
        n_gram (int): BLEU-n order (2 or 3 recommended for generation diversity)

    Returns:
        float: Average Self-BLEU score across all samples
    """

    # Apply smoothing to avoid zero BLEU scores for short or rare n-grams
    smoothie = SmoothingFunction().method1
    scores = []

    # Define BLEU n-gram weights
    # e.g., BLEU-2 uses equal weight for unigrams and bigrams
    weights = {
        2: (0.5, 0.5),
        3: (1/3, 1/3, 1/3),
        4: (0.25, 0.25, 0.25, 0.25)
    }[n_gram]

    # Tokenize all texts once for efficiency
    tokenized = [t.lower().split() for t in texts]

    # Compute Self-BLEU:
    # Each text is treated as the hypothesis,
    # while all other texts act as references
    for i, hypothesis in enumerate(tokenized):
        references = tokenized[:i] + tokenized[i+1:]
        score = sentence_bleu(
            references,
            hypothesis,
            weights=weights,
            smoothing_function=smoothie
        )
        scores.append(score)
    scores = np.array(scores)

    # Return the average Self-BLEU across all hypotheses, and the standard deviations
    if return_std:
        return scores.mean(), scores.std()
    else:
        return scores.mean()

In [11]:
def generate_samples(pipe, prompt, n=30):
    """
    Generate multiple stochastic text samples from a text-generation pipeline.

    This function is used to collect a set of outputs from the same model
    under identical decoding conditions, which enables statistical analysis
    of diversity, style, and surprisal.

    Args:
        pipe: Hugging Face text-generation pipeline
        prompt (str): Input prompt used for all generations
        n (int): Number of samples to generate

    Returns:
        list[str]: List of generated texts
    """
    return [
        pipe(
            prompt,
            max_new_tokens=200,       # maximum length of generated continuation
            min_new_tokens=100,       # enforce a minimum output length
            do_sample=True,           # enable stochastic sampling
            temperature=0.9,          # controls randomness/creativity
            top_p=0.95,               # nucleus sampling threshold
            #repetition_penalty=1.1,  # (optional) discourage repetition
            #no_repeat_ngram_size=3   # (optional) prevent repeating n-grams
        )[0]["generated_text"]        # extract generated text from pipeline output
        for _ in range(n)
    ]

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

def save_generations(texts, checkpoint, prompt, out_dir="generated_texts"):
    """
    Save generated texts to a JSON file for a given model checkpoint and prompt.

    Args:
        texts (list[str]): Generated text samples.
        checkpoint (str or int): Identifier of the model checkpoint.
        prompt (str): The prompt used to generate the texts.
        out_dir (str, optional): Directory where JSON files will be saved. Defaults to "generated_texts".

    Returns:
        Path: Path to the saved JSON file.
    """

    # Convert the output directory string to a Path object
    # and create it if it doesn't exist
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Specify the model used
    model_used = "apertus"

    # Create a dictionary containing all relevant information to save
    # This will be written to the JSON file
    data = {
        "model": model_used,       # Name of the model used
        "checkpoint": checkpoint,  # Checkpoint identifier
        "prompt": prompt,          # Prompt that was used to generate texts
        "n_samples": len(texts),   # Number of generated samples
        "texts": texts             # List of generated text strings
    }

    # Construct the full output file path
    # Filename includes model name and checkpoint for easy identification
    out_path = out_dir / f"generations_{model_used}_{checkpoint}.json"

    # Open the file in write mode with UTF-8 encoding and save the dictionary as JSON
    # indent=2 makes the file human-readable
    # ensure_ascii=False allows non-ASCII characters to be saved properly
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    # Return the Path object pointing to the saved file
    return out_path

### Generate texts for self-blue comparison

In [ ]:
baseline_texts = generate_samples(pipe_base, prompt, n=30)
save_generations(baseline_texts, "base", prompt)
mean_base, std_base = self_bleu(baseline_texts, n_gram=2, return_std=True)
print(f"Baseline Self-BLEU: mean={mean_base:.3f}, std={std_base:.3f}")


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


NameError: name 'np' is not defined

In [30]:
mean_base, std_base

(np.float64(0.4969483742685018), np.float64(0.06005965566229476))

In [27]:
# Clean up baseline model to free GPU memory
del pipe_base
torch.cuda.empty_cache()
gc.collect()
print("✓ Cleaned up baseline model memory")

✓ Cleaned up baseline model memory


In [28]:
tokenizer_epoch1 = AutoTokenizer.from_pretrained(checkpoint_paths[0])
model_epoch1 = AutoModelForCausalLM.from_pretrained(
        checkpoint_paths[0],
        torch_dtype=torch.bfloat16,  # Use bfloat16 to save memory (same as training)
        device_map="auto"             # Automatic device placement
    )


Loading checkpoint shards: 100%|██████████| 4/4 [00:21<00:00,  5.37s/it]


In [29]:
pipe_epoch1 = pipeline(
    "text-generation",
    model=model_epoch1,
    tokenizer=tokenizer_epoch1,
)

Device set to use cuda:0


In [31]:
epoch1_texts = generate_samples(pipe_epoch1, prompt, n=30)
save_generations(epoch1_texts, "epoch1", prompt)
mean_epoch1, std_epoch1 = self_bleu(epoch1_texts, n_gram=2, return_std=True)
print(f"Epoch1 Self-BLEU: mean={mean_epoch1:.3f}, std={std_epoch1:.3f}")

Epoch1 Self-BLEU: mean=0.544, std=0.035


In [32]:
# Clean up to free GPU memory before loading next checkpoint
del model_epoch1
del pipe_epoch1
del tokenizer_epoch1
torch.cuda.empty_cache()
gc.collect()
print("✓ Cleaned up epoch1 model memory")

✓ Cleaned up epoch1 model memory


In [33]:
tokenizer_epoch2 = AutoTokenizer.from_pretrained(checkpoint_paths[1])
model_epoch2 = AutoModelForCausalLM.from_pretrained(
        checkpoint_paths[1],
        torch_dtype=torch.bfloat16,  # Use bfloat16 to save memory (same as training)
        device_map="auto"             # Automatic device placement
    )


Loading checkpoint shards: 100%|██████████| 4/4 [00:22<00:00,  5.55s/it]


In [34]:
pipe_epoch2 = pipeline(
    "text-generation",
    model=model_epoch2,
    tokenizer=tokenizer_epoch2,
)

Device set to use cuda:0


In [35]:
epoch2_texts = generate_samples(pipe_epoch2, prompt, n=30)
save_generations(epoch2_texts, "epoch2", prompt)
mean_epoch2, std_epoch2 = self_bleu(epoch2_texts, n_gram=2, return_std=True)
print(f"Epoch2 Self-BLEU: mean={mean_epoch2:.3f}, std={std_epoch2:.3f}")

Epoch2 Self-BLEU: mean=0.533, std=0.046


In [36]:
# Clean up to free GPU memory before loading next checkpoint
del model_epoch2
del pipe_epoch2
del tokenizer_epoch2
torch.cuda.empty_cache()
gc.collect()
print("✓ Cleaned up epoch2 model memory")

✓ Cleaned up epoch2 model memory


In [37]:
tokenizer_epoch3 = AutoTokenizer.from_pretrained(checkpoint_paths[2])
model_epoch3 = AutoModelForCausalLM.from_pretrained(
        checkpoint_paths[2],
        torch_dtype=torch.bfloat16,  # Use bfloat16 to save memory (same as training)
        device_map="auto"             # Automatic device placement
    )

Loading checkpoint shards: 100%|██████████| 4/4 [00:23<00:00,  5.87s/it]


In [38]:
pipe_epoch3 = pipeline(
    "text-generation",
    model=model_epoch3,
    tokenizer=tokenizer_epoch3,
)

Device set to use cuda:0


In [39]:
epoch3_texts = generate_samples(pipe_epoch3, prompt, n=30)
save_generations(epoch3_texts, "epoch3", prompt)
mean_epoch3, std_epoch3 = self_bleu(epoch3_texts, n_gram=2, return_std=True)
print(f"Epoch3 Self-BLEU: mean={mean_epoch3:.3f}, std={std_epoch3:.3f}")

Epoch3 Self-BLEU: mean=0.543, std=0.039


In [40]:
# Clean up to free GPU memory before loading next checkpoint
del model_epoch3
del pipe_epoch3
del tokenizer_epoch3
torch.cuda.empty_cache()
gc.collect()
print("✓ Cleaned up epoch3 model memory")

✓ Cleaned up epoch3 model memory


In [41]:
tokenizer_epoch4 = AutoTokenizer.from_pretrained(checkpoint_paths[3])
model_epoch4 = AutoModelForCausalLM.from_pretrained(
        checkpoint_paths[3],
        torch_dtype=torch.bfloat16,  # Use bfloat16 to save memory (same as training)
        device_map="auto"             # Automatic device placement
    )

Loading checkpoint shards: 100%|██████████| 4/4 [00:25<00:00,  6.31s/it]


In [42]:
pipe_epoch4 = pipeline(
    "text-generation",
    model=model_epoch4,
    tokenizer=tokenizer_epoch4,
)

Device set to use cuda:0


In [43]:
epoch4_texts = generate_samples(pipe_epoch4, prompt, n=30)
save_generations(epoch4_texts, "epoch4", prompt)
mean_epoch4, std_epoch4 = self_bleu(epoch4_texts, n_gram=2, return_std=True)
print(f"Epoch4 Self-BLEU: mean={mean_epoch4:.3f}, std={std_epoch4:.3f}")

Epoch4 Self-BLEU: mean=0.547, std=0.062


In [44]:
# Clean up to free GPU memory before loading next checkpoint
del model_epoch4
del pipe_epoch4
del tokenizer_epoch4
torch.cuda.empty_cache()
gc.collect()
print("✓ Cleaned up epoch4 model memory")

✓ Cleaned up epoch4 model memory


In [45]:
tokenizer_epoch5 = AutoTokenizer.from_pretrained(checkpoint_paths[4])
model_epoch5 = AutoModelForCausalLM.from_pretrained(
        checkpoint_paths[4],
        torch_dtype=torch.bfloat16,  # Use bfloat16 to save memory (same as training)
        device_map="auto"             # Automatic device placement
    )

Loading checkpoint shards: 100%|██████████| 4/4 [00:21<00:00,  5.35s/it]


In [46]:
pipe_epoch5 = pipeline(
    "text-generation",
    model=model_epoch5,
    tokenizer=tokenizer_epoch5,
)

Device set to use cuda:0


In [47]:
epoch5_texts = generate_samples(pipe_epoch5, prompt, n=30)
save_generations(epoch5_texts, "epoch5", prompt)
mean_epoch5, std_epoch5 = self_bleu(epoch5_texts, n_gram=2, return_std=True)
print(f"Epoch5 Self-BLEU: mean={mean_epoch5:.3f}, std={std_epoch5:.3f}")

Epoch5 Self-BLEU: mean=0.545, std=0.058


In [48]:
# Clean up to free GPU memory before loading next checkpoint
del model_epoch5
del pipe_epoch5
del tokenizer_epoch5
torch.cuda.empty_cache()
gc.collect()
print("✓ Cleaned up epoch5 model memory")

✓ Cleaned up epoch5 model memory


In [49]:
import pandas as pd

In [54]:
self_blue_df = pd.DataFrame({
    'checkpoint': ["base", "epoch1", "epoch2", "epoch3", "epoch4", "epoch5"],
    'mean': [mean_base, mean_epoch1, mean_epoch2, mean_epoch3, mean_epoch4, mean_epoch5],
    'std': [std_base, std_epoch1, std_epoch2, std_epoch3, std_epoch4, std_epoch5]
})

In [55]:
self_blue_df

,checkpoint,mean,std
0,base,0.496948,0.060060
1,epoch1,0.544087,0.034643
2,epoch2,0.533150,0.046061
3,epoch3,0.542873,0.038807
4,epoch4,0.547236,0.062192
5,epoch5,0.545036,0.057853


In [56]:
self_bleu_df.to_csv("apertus_self_blue.csv", index=False)

---

## Lexical Diversity

Higher = more varied phrasing

Distinct-n measures lexical diversity by computing the proportion of unique n-word sequences (n-grams) in a text, where higher values indicate less repetitive phrasing but do not capture semantic quality or true creativity, making it most useful for relative comparison (e.g., before vs. after fine-tuning) on texts of similar length.

In [58]:
def distinct_n(texts, n=2, return_std=True):
    """
    Compute Distinct-n to measure lexical diversity in generated text.
    Optionally returns standard deviation across texts.

    Args:
        texts (list[str] or str): Generated texts (single text or list of texts)
        n (int): Size of n-grams (e.g., 2 for bigrams)

    Returns:
        float: Distinct-n score
    """
    # Allow single string input by converting it to a list
    if isinstance(texts, str):
        texts = [texts]

    scores = []

    # Extract n-grams from all texts
    for text in texts:
        tokens = text.lower().split()

        # Generate n-grams using a sliding window
        ngrams = list(zip(*[tokens[i:] for i in range(n)]))

        total = len(ngrams)             # Total number of n-grams in this text
        unique = len(set(ngrams))       # Number of unique n-grams
        score = unique / max(1, total)  # Distinct-n score for this text
        scores.append(score)            # Add score to list
        
    scores = np.array(scores)           # Convert list of scores to numpy array for easy statistics
        
    # Return mean (and optionally std) of Distinct-n across all texts
    if return_std:
        return scores.mean(), scores.std()
    else:
        return scores.mean()

In [61]:
dis_mean_base, dis_std_base = distinct_n(baseline_texts, n=2, return_std=True)
print(f"Distinct-2: mean={dis_mean_base:.3f}, std={dis_std_base:.3f}")

dis_mean_epoch1, dis_std_epoch1 = distinct_n(epoch1_texts, n=2, return_std=True)
print(f"Distinct-2: mean={dis_mean_epoch1:.3f}, std={dis_std_epoch1:.3f}")

dis_mean_epoch2, dis_std_epoch2 = distinct_n(epoch2_texts, n=2, return_std=True)
print(f"Distinct-2: mean={dis_mean_epoch2:.3f}, std={dis_std_epoch2:.3f}")

dis_mean_epoch3, dis_std_epoch3 = distinct_n(epoch3_texts, n=2, return_std=True)
print(f"Distinct-2: mean={dis_mean_epoch3:.3f}, std={dis_std_epoch3:.3f}")

dis_mean_epoch4, dis_std_epoch4 = distinct_n(epoch4_texts, n=2, return_std=True)
print(f"Distinct-2: mean={dis_mean_epoch4:.3f}, std={dis_std_epoch4:.3f}")

dis_mean_epoch5, dis_std_epoch5 = distinct_n(epoch5_texts, n=2, return_std=True)
print(f"Distinct-2: mean={dis_mean_epoch5:.3f}, std={dis_std_epoch5:.3f}")

Distinct-2: mean=0.894, std=0.061
Distinct-2: mean=0.935, std=0.026
Distinct-2: mean=0.937, std=0.029
Distinct-2: mean=0.939, std=0.025
Distinct-2: mean=0.925, std=0.037
Distinct-2: mean=0.936, std=0.024


In [63]:
lexical_diverse_df = pd.DataFrame({
    "checkpoint": ["base", "epoch1", "epoch2", "epoch3", "epoch4", "epoch5"],
    "mean": [dis_mean_base, dis_mean_epoch1, dis_mean_epoch2, dis_mean_epoch3, dis_mean_epoch4, dis_mean_epoch5],
    "std": [dis_std_base, dis_std_epoch1, dis_std_epoch2, dis_std_epoch3, dis_std_epoch4, dis_std_epoch5]
})

In [64]:
lexical_diverse_df

,checkpoint,mean,std
0,base,0.893511,0.061460
1,epoch1,0.934830,0.025682
2,epoch2,0.937359,0.029389
3,epoch3,0.938846,0.025309
4,epoch4,0.924992,0.036690
5,epoch5,0.935733,0.024256


In [65]:
lexical_diverse_df.to_csv("apertus_lexical_diverse.csv", index=False)

---

## Surprisal (novelty vs generic English)

Slightly higher surprisal after fine-tuning = more novelty  
Too high = incoherent

This code measures surprisal by computing the model’s average negative log-likelihood (cross-entropy loss) over all tokens in the given text, which reflects how unexpected the text is to the model: higher loss means the model assigns lower probability to the observed tokens (higher surprisal), while lower loss means the text is more predictable according to the model.

In [ ]:
def surprisal(model, tokenizer, texts, return_std=True):
    """
    Compute token-level surprisal (negative log-likelihood) of text(s)
    under a given language model.

    Args:
        model: Causal language model used to compute likelihoods
        tokenizer: Corresponding tokenizer
        texts (str or list[str]): Text(s) to evaluate

    Returns:
        float:
            - Surprisal for a single text
            - Token-length-weighted average surprisal for multiple texts
    """
    # Allow single string input by converting it to a list
    if isinstance(texts, str):
        texts = [texts]

    text_losses = []      # Stores total negative log-likelihood (loss * tokens) per text
    text_lengths = []     # Stores number of tokens per text

    # Set model to evaluation mode (disables dropout, etc.)
    model.eval()

    # Disable gradient computation for efficient inference
    with torch.no_grad():
        for text in texts:

            # Tokenize input text and move tensors to the model device
            inputs = tokenizer(text, return_tensors="pt").to(model.device)
            
            # Compute negative log-likelihood by predicting each token
            # conditioned on previous tokens
            outputs = model(**inputs, labels=inputs["input_ids"])

            # outputs.loss is the mean loss over all tokens in the sequence
            loss = outputs.loss.item()

            # Number of tokens in the sequence
            n_tokens = inputs["input_ids"].numel()

            # Store token-weighted loss for this text
            text_losses.append(loss * n_tokens)
            text_lengths.append(n_tokens)

    # Convert lists to NumPy arrays for vectorized computation
    text_losses = np.array(text_losses)
    text_lengths = np.array(text_lengths)

    # Compute token-length-weighted average surprisal across all texts
    weighted_mean = text_losses.sum() / text_lengths.sum()

    if return_std:
        # Compute weighted standard deviation across texts
        # weights proportional to number of tokens per text
        weights = text_lengths / text_lengths.sum()

         # Compute squared deviation of each text's average surprisal from the overall weighted mean
        weighted_var = np.sum(weights * (text_losses / text_lengths - weighted_mean)**2)
        weighted_std = np.sqrt(weighted_var)

        # Return both mean and std
        return weighted_mean, weighted_std

    return weighted_mean

### Reload baseline model for surprisal evaluation

In [69]:
# Reload baseline model and tokenizer for surprisal evaluation
tokenizer_base = AutoTokenizer.from_pretrained("swiss-ai/Apertus-8B-2509")
model_base = AutoModelForCausalLM.from_pretrained(
    "swiss-ai/Apertus-8B-2509",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:22<00:00,  5.61s/it]


In [70]:
# Evaluate surprisal of baseline texts using baseline model
surp_mean_base, surp_std_base = surprisal(
    model_base, 
    tokenizer_base, 
    baseline_texts, 
    return_std=True
)

In [71]:
# Evaluate surprisal of all epoch texts using baseline model as reference
surp_mean_epoch1, surp_std_epoch1 = surprisal(model_base, tokenizer_base, epoch1_texts, return_std=True)
print(f"Epoch1 Surprisal: mean={surp_mean_epoch1:.3f}, std={surp_std_epoch1:.3f}")

surp_mean_epoch2, surp_std_epoch2 = surprisal(model_base, tokenizer_base, epoch2_texts, return_std=True)
print(f"Epoch2 Surprisal: mean={surp_mean_epoch2:.3f}, std={surp_std_epoch2:.3f}")

surp_mean_epoch3, surp_std_epoch3 = surprisal(model_base, tokenizer_base, epoch3_texts, return_std=True)
print(f"Epoch3 Surprisal: mean={surp_mean_epoch3:.3f}, std={surp_std_epoch3:.3f}")

surp_mean_epoch4, surp_std_epoch4 = surprisal(model_base, tokenizer_base, epoch4_texts, return_std=True)
print(f"Epoch4 Surprisal: mean={surp_mean_epoch4:.3f}, std={surp_std_epoch4:.3f}")

surp_mean_epoch5, surp_std_epoch5 = surprisal(model_base, tokenizer_base, epoch5_texts, return_std=True)
print(f"Epoch5 Surprisal: mean={surp_mean_epoch5:.3f}, std={surp_std_epoch5:.3f}")

Epoch1 Surprisal: mean=1.646, std=0.164
Epoch2 Surprisal: mean=1.620, std=0.146
Epoch3 Surprisal: mean=1.627, std=0.169
Epoch4 Surprisal: mean=1.652, std=0.166
Epoch5 Surprisal: mean=1.658, std=0.151


In [72]:
surprisal_df = pd.DataFrame({
    "checkpoint": ["base", "epoch1", "epoch2", "epoch3", "epoch4", "epoch5"],
    "mean": [surp_mean_base, surp_mean_epoch1, surp_mean_epoch2, surp_mean_epoch3, surp_mean_epoch4, surp_mean_epoch5],
    "std": [surp_std_base, surp_std_epoch1, surp_std_epoch2, surp_std_epoch3, surp_std_epoch4, surp_std_epoch5]
})
surprisal_df

,checkpoint,mean,std
0,base,1.461742,0.168021
1,epoch1,1.646491,0.164344
2,epoch2,1.620066,0.145767
3,epoch3,1.627379,0.168592
4,epoch4,1.652152,0.165833
5,epoch5,1.658184,0.151413


In [73]:
surprisal_df.to_csv("apertus_surprisal.csv", index=False)

---

## Style alignment (fairy-tale vocabulary)

This code measures style alignment by counting how many distinct, predefined fairy-tale–related keywords appear in the text, using their presence as a simple proxy for how closely the text matches a fairy-tale style, without considering context, frequency, or deeper narrative structure.

In [74]:
fairy_words = {
    # Core beings & roles
    "fairy", "fairies", "king", "queen", "prince", "princess",
    "witch", "wizard", "sorcerer", "sorceress", "mage",
    "giant", "ogre", "dragon", "elf", "dwarf", "goblin",
    "knight", "hero", "villain",

    # Magic & supernatural
    "magic", "magical", "spell", "spells", "curse", "cursed",
    "enchantment", "enchanted", "charm", "potion",
    "wand", "crystal", "prophecy", "destiny",

    # Places & settings
    "forest", "woods", "kingdom", "castle", "tower",
    "palace", "village", "cottage", "hut",
    "mountain", "cave", "river", "lake",

    # Objects & symbols
    "crown", "throne", "sword", "shield", "cloak",
    "ring", "mirror", "key", "treasure", "gold",
    "apple", "rose", "needle",

    # Fairy-tale actions & themes
    "wish", "wished", "dream", "dreamed",
    "journey", "quest", "trial",
    "promise", "forbidden",
    "curse", "blessing",
    "transformed", "transformation",

    # Narrative markers
    "once", "kingdom", "ancient", "mystical",
    "legend", "tale", "story", "myth"
}

In [75]:
def fairy_score(texts, return_std=True):
    """
    Compute a simple "fairy-word" score to measure fairy-tale style.

    - For a single string, returns the number of fairy words present.
    - For a list of strings, returns the average number of fairy words per text.

    Args:
        texts (str or list[str]): Generated text(s) to evaluate

    Returns:
        float: Average number of fairy words
    """
    # Allow single string input
    if isinstance(texts, str):
        texts = [texts]

    scores = []

    for text in texts:
        
        # Convert text to lowercase tokens
        tokens = set(text.lower().split())

        # Count overlap with the fairy_words set
        scores.append(len(tokens & fairy_words))

    mean_score = np.mean(scores)

    if return_std:
        std_score = np.std(scores, ddof=0)
        return mean_score, std_score

    return mean_score

In [76]:
# Evaluate fairy-tale style alignment across all checkpoints
fairy_mean_base, fairy_std_base = fairy_score(baseline_texts, return_std=True)
print(f"Baseline Fairy Score: mean={fairy_mean_base:.3f}, std={fairy_std_base:.3f}")

fairy_mean_epoch1, fairy_std_epoch1 = fairy_score(epoch1_texts, return_std=True)
print(f"Epoch1 Fairy Score: mean={fairy_mean_epoch1:.3f}, std={fairy_std_epoch1:.3f}")

fairy_mean_epoch2, fairy_std_epoch2 = fairy_score(epoch2_texts, return_std=True)
print(f"Epoch2 Fairy Score: mean={fairy_mean_epoch2:.3f}, std={fairy_std_epoch2:.3f}")

fairy_mean_epoch3, fairy_std_epoch3 = fairy_score(epoch3_texts, return_std=True)
print(f"Epoch3 Fairy Score: mean={fairy_mean_epoch3:.3f}, std={fairy_std_epoch3:.3f}")

fairy_mean_epoch4, fairy_std_epoch4 = fairy_score(epoch4_texts, return_std=True)
print(f"Epoch4 Fairy Score: mean={fairy_mean_epoch4:.3f}, std={fairy_std_epoch4:.3f}")

fairy_mean_epoch5, fairy_std_epoch5 = fairy_score(epoch5_texts, return_std=True)
print(f"Epoch5 Fairy Score: mean={fairy_mean_epoch5:.3f}, std={fairy_std_epoch5:.3f}")

Baseline Fairy Score: mean=4.367, std=1.683
Epoch1 Fairy Score: mean=3.833, std=1.344
Epoch2 Fairy Score: mean=3.300, std=1.242
Epoch3 Fairy Score: mean=3.267, std=1.263
Epoch4 Fairy Score: mean=3.767, std=1.606
Epoch5 Fairy Score: mean=3.467, std=1.176


In [77]:
fairy_df = pd.DataFrame({
    "checkpoint": ["base", "epoch1", "epoch2", "epoch3", "epoch4", "epoch5"],
    "mean": [fairy_mean_base, fairy_mean_epoch1, fairy_mean_epoch2, fairy_mean_epoch3, fairy_mean_epoch4, fairy_mean_epoch5],
    "std": [fairy_std_base, fairy_std_epoch1, fairy_std_epoch2, fairy_std_epoch3, fairy_std_epoch4, fairy_std_epoch5]
})
fairy_df

,checkpoint,mean,std
0,base,4.366667,1.682921
1,epoch1,3.833333,1.343710
2,epoch2,3.300000,1.242310
3,epoch3,3.266667,1.263153
4,epoch4,3.766667,1.605892
5,epoch5,3.466667,1.175679


In [78]:
fairy_df.to_csv("apertus_fairy_score.csv", index=False)

## Classifier Model as a Judge

In [79]:
# Create a zero-shot text classifier using Hugging Face pipeline
# This classifier will be used to estimate how "fairy-tale-ish" a text is
# without any additional training on fairy-tale data.
#
# Model: facebook/bart-large-mnli
#   - Pretrained for natural language inference (NLI)
#   - Can perform zero-shot classification by checking entailment
#     between a text and candidate labels

fairytale_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device="cuda"
)

Device set to use cuda


In [80]:
def fairytale_scores_zs(texts, batch_size=8):
    """
    Compute zero-shot "fairy-tale" probabilities for a list of texts using a pretrained NLI model.

    Args:
        texts (list[str]): List of texts to evaluate
        batch_size (int): Number of texts to process in parallel (for efficiency)

    Returns:
        list[float]: Probability of "fairy tale" for each input text
    """

    # Run the zero-shot classifier
    # Candidate labels cover fairy tales and other genres as distractors
    # hypothesis_template allows the NLI model to interpret the labels
    results = fairytale_classifier(
        texts,
        candidate_labels=[
            "fairy tale",
            "news article",
            "scientific text",
            "modern novel"
        ],
        hypothesis_template="This text is a {}.",
        batch_size=batch_size
    )

    # Transformers returns a list of dicts with labels and scores
    # Extract the probability corresponding to "fairy tale"
    scores = []
    for r in results:
        label_scores = dict(zip(r["labels"], r["scores"]))
        scores.append(label_scores["fairy tale"])

    return scores

In [81]:
# Evaluate zero-shot fairy-tale classification across all checkpoints
zs_scores_base = fairytale_scores_zs(baseline_texts, batch_size=8)
zs_mean_base, zs_std_base = np.mean(zs_scores_base), np.std(zs_scores_base)
print(f"Baseline ZS Fairy Score: mean={zs_mean_base:.3f}, std={zs_std_base:.3f}")

zs_scores_epoch1 = fairytale_scores_zs(epoch1_texts, batch_size=8)
zs_mean_epoch1, zs_std_epoch1 = np.mean(zs_scores_epoch1), np.std(zs_scores_epoch1)
print(f"Epoch1 ZS Fairy Score: mean={zs_mean_epoch1:.3f}, std={zs_std_epoch1:.3f}")

zs_scores_epoch2 = fairytale_scores_zs(epoch2_texts, batch_size=8)
zs_mean_epoch2, zs_std_epoch2 = np.mean(zs_scores_epoch2), np.std(zs_scores_epoch2)
print(f"Epoch2 ZS Fairy Score: mean={zs_mean_epoch2:.3f}, std={zs_std_epoch2:.3f}")

zs_scores_epoch3 = fairytale_scores_zs(epoch3_texts, batch_size=8)
zs_mean_epoch3, zs_std_epoch3 = np.mean(zs_scores_epoch3), np.std(zs_scores_epoch3)
print(f"Epoch3 ZS Fairy Score: mean={zs_mean_epoch3:.3f}, std={zs_std_epoch3:.3f}")

zs_scores_epoch4 = fairytale_scores_zs(epoch4_texts, batch_size=8)
zs_mean_epoch4, zs_std_epoch4 = np.mean(zs_scores_epoch4), np.std(zs_scores_epoch4)
print(f"Epoch4 ZS Fairy Score: mean={zs_mean_epoch4:.3f}, std={zs_std_epoch4:.3f}")

zs_scores_epoch5 = fairytale_scores_zs(epoch5_texts, batch_size=8)
zs_mean_epoch5, zs_std_epoch5 = np.mean(zs_scores_epoch5), np.std(zs_scores_epoch5)
print(f"Epoch5 ZS Fairy Score: mean={zs_mean_epoch5:.3f}, std={zs_std_epoch5:.3f}")

Baseline ZS Fairy Score: mean=0.721, std=0.130
Epoch1 ZS Fairy Score: mean=0.817, std=0.077
Epoch2 ZS Fairy Score: mean=0.823, std=0.067
Epoch3 ZS Fairy Score: mean=0.823, std=0.061
Epoch4 ZS Fairy Score: mean=0.822, std=0.065
Epoch5 ZS Fairy Score: mean=0.818, std=0.053


In [82]:
zs_fairy_df = pd.DataFrame({
    "checkpoint": ["base", "epoch1", "epoch2", "epoch3", "epoch4", "epoch5"],
    "mean": [zs_mean_base, zs_mean_epoch1, zs_mean_epoch2, zs_mean_epoch3, zs_mean_epoch4, zs_mean_epoch5],
    "std": [zs_std_base, zs_std_epoch1, zs_std_epoch2, zs_std_epoch3, zs_std_epoch4, zs_std_epoch5]
})
zs_fairy_df

,checkpoint,mean,std
0,base,0.721023,0.129517
1,epoch1,0.816881,0.076936
2,epoch2,0.823474,0.067149
3,epoch3,0.823021,0.060725
4,epoch4,0.822401,0.064503
5,epoch5,0.818254,0.053413


In [83]:
zs_fairy_df.to_csv("apertus_zs_fairy_score.csv", index=False)

# Summary table

In [86]:
# Merge all metrics into one comprehensive table
summary_df = self_blue_df.copy()
summary_df = summary_df.rename(columns={'mean': 'self_bleu_mean', 'std': 'self_bleu_std'})

summary_df = summary_df.merge(
    lexical_diverse_df.rename(columns={'mean': 'distinct_2_mean', 'std': 'distinct_2_std'}),
    on='checkpoint'
)

summary_df = summary_df.merge(
    surprisal_df.rename(columns={'mean': 'surprisal_mean', 'std': 'surprisal_std'}),
    on='checkpoint'
)

summary_df = summary_df.merge(
    fairy_df.rename(columns={'mean': 'fairy_word_mean', 'std': 'fairy_word_std'}),
    on='checkpoint'
)

summary_df = summary_df.merge(
    zs_fairy_df.rename(columns={'mean': 'fairy_classifier_mean', 'std': 'fairy_classifier_std'}),
    on='checkpoint'
)

summary_df

,checkpoint,self_bleu_mean,self_bleu_std,distinct_2_mean,distinct_2_std,surprisal_mean,surprisal_std,fairy_word_mean,fairy_word_std,fairy_classifier_mean,fairy_classifier_std
0,base,0.496948,0.060060,0.893511,0.061460,1.461742,0.168021,4.366667,1.682921,0.721023,0.129517
1,epoch1,0.544087,0.034643,0.934830,0.025682,1.646491,0.164344,3.833333,1.343710,0.816881,0.076936
2,epoch2,0.533150,0.046061,0.937359,0.029389,1.620066,0.145767,3.300000,1.242310,0.823474,0.067149
3,epoch3,0.542873,0.038807,0.938846,0.025309,1.627379,0.168592,3.266667,1.263153,0.823021,0.060725
4,epoch4,0.547236,0.062192,0.924992,0.036690,1.652152,0.165833,3.766667,1.605892,0.822401,0.064503
5,epoch5,0.545036,0.057853,0.935733,0.024256,1.658184,0.151413,3.466667,1.175679,0.818254,0.053413


In [87]:
# Save the comprehensive summary table
summary_df.to_csv("apertus_8B.csv", index=False)